In [1]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from dotenv import load_dotenv
from langchain_classic.retrievers import MultiQueryRetriever
load_dotenv()

True

In [2]:
#relevant health and wellness documents

all_docs = [
    Document(page_content="A balanced diet rich in fruits, vegetables, and whole grains supports long-term health."),
    Document(page_content="Regular exercise, including cardio and strength training, boosts energy levels and overall fitness."),
    Document(page_content="Adequate sleep and stress management are essential for maintaining physical and mental well-being."),
    Document(page_content="Staying hydrated throughout the day helps regulate body temperature and supports organ function."),
    Document(page_content="Avoiding processed foods and excessive sugar intake can reduce the risk of chronic diseases."),
    Document(page_content="The solar system in modern homes helps balance electricity demand by using renewable energy."),
    Document(page_content="Python is a versatile programming language used in web development, data science, and automation."),
    Document(page_content="Photosynthesis is the process by which plants convert sunlight into chemical energy."),
    Document(page_content="The FIFA World Cup is one of the most-watched sporting events globally, held every four years."),
]

In [3]:
#embedding model
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

#create FAISS vector store
vector_store=(FAISS.from_documents(documents=all_docs, embedding=embedding))

In [4]:
#create retrievers (simple retriever)
similarity_retriever=vector_store.as_retriever(search_type='similarity', search_kwargs={'k':5})


In [5]:
llm=HuggingFaceEndpoint(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    task= "text-generation"
)

model= ChatHuggingFace(llm=llm)

#multiquery_retriever
multiquery_retriever=MultiQueryRetriever.from_llm(
    retriever=vector_store.as_retriever(search_kwargs={'k':5}),
        llm=model
)

In [6]:
#query
query='How to improve energy levels and maintain balance?'

In [10]:
#retriever results
similarity_results=similarity_retriever.invoke(query)
multiquery_results=multiquery_retriever.invoke(query)
top_n=5 #retestrict the results to 5 only
multiquery_results = multiquery_results[:top_n]


In [11]:
#results
#print retrieved content
for i,doc in enumerate(similarity_results):
    print(f"\n ----Result {i+1}---- ")
    print(f"Content:\n{doc.page_content}---")


 ----Result 1---- 
Content:
Regular exercise, including cardio and strength training, boosts energy levels and overall fitness.---

 ----Result 2---- 
Content:
A balanced diet rich in fruits, vegetables, and whole grains supports long-term health.---

 ----Result 3---- 
Content:
Adequate sleep and stress management are essential for maintaining physical and mental well-being.---

 ----Result 4---- 
Content:
The solar system in modern homes helps balance electricity demand by using renewable energy.---

 ----Result 5---- 
Content:
Staying hydrated throughout the day helps regulate body temperature and supports organ function.---


In [12]:
#print retrieved content
for i,doc in enumerate(multiquery_results):
    print(f"\n ----Result {i+1}---- ")
    print(f"Content:\n{doc.page_content}---")


 ----Result 1---- 
Content:
A balanced diet rich in fruits, vegetables, and whole grains supports long-term health.---

 ----Result 2---- 
Content:
Regular exercise, including cardio and strength training, boosts energy levels and overall fitness.---

 ----Result 3---- 
Content:
Avoiding processed foods and excessive sugar intake can reduce the risk of chronic diseases.---

 ----Result 4---- 
Content:
Staying hydrated throughout the day helps regulate body temperature and supports organ function.---

 ----Result 5---- 
Content:
Adequate sleep and stress management are essential for maintaining physical and mental well-being.---
